# 🪸 Servidor de Segmentação SAM ViT-H na Nuvem (Google Colab)
### Pipeline Híbrido: Meta Segment Anything (SAM ViT-H) + CoralNet

Este notebook executa o modelo **SAM ViT-H (Huge - 2.5 GB)** utilizando a GPU gratuita (Tesla T4 de 16 GB) do Google Colab e expõe uma API pública via túnel seguro.

Seu computador local (`d:\coralnet-sam`) envia as fotos para cá, recebe os polígonos perfeitos de cada colônia e os classifica com o CoralNet!

In [ ]:
# 1. Verificar GPU alocada pelo Google Colab
!nvidia-smi

In [ ]:
# 2. Instalar Segment Anything da Meta e servidor FastAPI
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q fastapi uvicorn pyngrok nest_asyncio python-multipart pillow opencv-python pycocotools


In [ ]:
# 3. Baixar os pesos do modelo SAM ViT-H (2.5 GB) na NVMe do Colab
import os
import urllib.request

WEIGHTS_URL = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
WEIGHTS_FILE = "sam_vit_h_4b8939.pth"

if not os.path.exists(WEIGHTS_FILE):
    print("Baixando sam_vit_h_4b8939.pth (2.5 GB)... Aguarde cerca de 30 segundos.")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_FILE)
    print("Download concluído com sucesso!")
else:
    print("Pesos do SAM ViT-H já presentes no ambiente.")

In [ ]:
# 4. Carregar o SAM na GPU e configurar o gerador automático de máscaras para corais
import torch
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Carregando SAM ViT-H na GPU: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})...")

sam = sam_model_registry["vit_h"](checkpoint=WEIGHTS_FILE)
sam.to(device=device)

# Hiperparâmetros otimizados para imagens de recifes de corais (evita microfragmentos)
mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,            # Grade densa de 1024 pontos para descobrir todas as colônias
    pred_iou_thresh=0.86,          # Qualidade do contorno da máscara
    stability_score_thresh=0.90,   # Estabilidade de borda sob corte
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=200,      # Ignora microfragmentos menores que 200 pixels
)

print("SAM ViT-H pronto para segmentar todas as colônias!")

In [ ]:
# 5. Criar a API FastAPI para receber as imagens do seu computador
from fastapi import FastAPI, File, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import io
import cv2
import numpy as np
from pycocotools import mask as mask_utils

app = FastAPI(title="SAM Coral Segmentation Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/health")
def health():
    return {
        "status": "online",
        "model": "SAM ViT-H (2.5GB)",
        "device": device,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    }

@app.post("/segment")
async def segment_image(file: UploadFile = File(...), min_area: int = 300, pred_iou_thresh: float = 0.86):
    # Ajustar dinamicamente hiperparametros no SAM antes de segmentar
    mask_generator.min_mask_region_area = int(min_area)
    mask_generator.pred_iou_thresh = float(pred_iou_thresh)
    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")
    image_np = np.array(image)
    h, w, _ = image_np.shape

    # Rodar SAM Segment Everything
    raw_masks = mask_generator.generate(image_np)

    masks_output = []
    for idx, m in enumerate(raw_masks, 1):
        bbox = [int(v) for v in m["bbox"]] # [x, y, w, h]
        area = int(m["area"])
        cx = bbox[0] + bbox[2] // 2
        cy = bbox[1] + bbox[3] // 2

        # Fase 2: Extrair poligonos simplificados com OpenCV
        seg_uint8 = m["segmentation"].astype(np.uint8)
        contours, _ = cv2.findContours(seg_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        simplified_polys = []
        for cnt in contours:
            if len(cnt) >= 3:
                approx = cv2.approxPolyDP(cnt, epsilon=1.5, closed=True)
                if len(approx) >= 3:
                    simplified_polys.append(approx.reshape(-1, 2).tolist())

        # Fase 2: Codificacao RLE compacta
        rle = mask_utils.encode(np.asfortranarray(m["segmentation"]))
        rle_dict = {"size": rle["size"], "counts": rle["counts"].decode("utf-8")}

        masks_output.append({
            "id": idx,
            "bbox": bbox,
            "centroid": [cy, cx],
            "area": area,
            "area_pct": (area / (w * h)) * 100.0,
            "polygon": simplified_polys[0] if simplified_polys else [],
            "polygons": simplified_polys,
            "rle": rle_dict,
            "predicted_iou": float(m["predicted_iou"]),
            "stability_score": float(m["stability_score"])
        })

    return {
        "image_width": w,
        "image_height": h,
        "total_masks": len(masks_output),
        "masks": masks_output
    }


In [ ]:
# 6. Abrir o túnel público e iniciar o servidor
# (Insira seu authtoken gratuito do ngrok.com abaixo se tiver, ou use localtunnel)
from pyngrok import ngrok
import nest_asyncio
import uvicorn

# Obtenha seu token gratuito em https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "" # Cole seu token do ngrok aqui se tiver

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Abrir túnel para a porta 8000
public_url = ngrok.connect(8000).public_url
print("\n" + "="*65)
print(f"🔥 SERVIDOR SAM ON-LINE!")
print(f"Copie esta URL e cole no seu projeto local (no config.py ou na interface):")
print(f"👉 {public_url}")
print("="*65 + "\n")

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)